Used dtataset: **House Prices - Advanced Regression Techniques**

In [3]:
import kagglehub
import pandas as pd

kagglehub.login()

In [ ]:
path = kagglehub.competition_download('house-prices-advanced-regression-techniques', output_dir='./data')

print("Path to files:", path)

100%|██████████| 199k/199k [00:00<00:00, 344kB/s]

Extracting files...
Path to competition files: ./data


In [13]:
train_df = pd.read_csv('./data/train.csv')
test_df = pd.read_csv('./data/test.csv')

print(train_df.shape)  # (1460, 81)
train_df.head()

(1460, 81)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [37]:
train_df.info()
test_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   str    
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   str    
 6   Alley          91 non-null     str    
 7   LotShape       1460 non-null   str    
 8   LandContour    1460 non-null   str    
 9   Utilities      1460 non-null   str    
 10  LotConfig      1460 non-null   str    
 11  LandSlope      1460 non-null   str    
 12  Neighborhood   1460 non-null   str    
 13  Condition1     1460 non-null   str    
 14  Condition2     1460 non-null   str    
 15  BldgType       1460 non-null   str    
 16  HouseStyle     1460 non-null   str    
 17  OverallQual    1460 non-null   int64  
 18  OverallCond    1460

# Przygotowanie danych

* zamienienie wartosci NaN na 0 itp
* Zamiana Stringów
* Standaryzacja (StandardScaler)
* Usuwanie outliersów (Z-score, IQR method, visually)

In [ ]:
def clean_data(df:pd.DataFrame):
    # Fill categorical "absent" columns with 'None'
    none_cols = ['Alley', 'PoolQC', 'MiscFeature', 'Fence', 'FireplaceQu',
                'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
                'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 
                'BsmtFinType2', 'Utilities', 'MasVnrType']
    df[none_cols] = df[none_cols].fillna('None')

    # Fill numeric absent columns with 0
    zero_cols = ['GarageArea', 'GarageCars', 'BsmtFinSF1', 'BsmtFinSF2', 
                'BsmtUnfSF', 'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath',
                'MasVnrArea']
    df[zero_cols] = df[zero_cols].fillna(0)

    # Fill truly missing numeric with median
    df['LotFrontage'] = df['LotFrontage'].fillna(df['LotFrontage'].median())

    ordinal_cols = {
    'Street':      ['Grvl', 'Pave'],
    'Alley':       ['None', 'Grvl', 'Pave'],
    'LotShape':    ['Reg', 'IR1', 'IR2', 'IR3'],
    'Utilities':   ['AllPub', 'NoSewr', 'NoSeWa', 'ELO'],
    'LandSlope':   ['Gtl', 'Mod', 'Sev'],
    'ExterQual':   ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'ExterCond':   ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtQual':    ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtCond':    ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtExposure':    ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],

    'KitchenQual': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'GarageQual':  ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'HeatingQC':   ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    }

    non_ordinal_cols = ['MSZoning','LandContour','LotConfig','Neighborhood',
                        'Condition1','Condition2','BldgType','HouseStyle',
                        'RoofStyle','RoofMatl','Exterior1st', 'Exterior2nd',
                        'MasVnrType','Foundation']

    int_cols_to_categories = ['MSSubClass']

    

# for col, order in ordinal_cols.items():
#     enc = OrdinalEncoder(categories=[order])
#     df[col] = enc.fit_transform(df[[col]])

    

In [ ]:
train_df.groupby('BsmtExposure')['SalePrice'].mean().sort_values()


BsmtFinType1
Rec    146889.248120
BLQ    149493.655405
LwQ    151852.702703
ALQ    161573.068182
Unf    170670.576744
GLQ    235413.720096
Name: SalePrice, dtype: float64

In [6]:
clean_data(train_df)
train_df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,None,Reg,Lvl,AllPub,...,0,None,None,None,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,None,Reg,Lvl,AllPub,...,0,None,None,None,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,None,IR1,Lvl,AllPub,...,0,None,None,None,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,None,IR1,Lvl,AllPub,...,0,None,None,None,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,None,IR1,Lvl,AllPub,...,0,None,None,None,0,12,2008,WD,Normal,250000


# Wizualizacja danych

* Wykresy pokazujące zależność między cechami a ceną. 
* TABELE
* INNE WYKRESY